# Fastest Reliable Text → JSON Enrichment with Qwen3-8B-AWQ + vLLM

This notebook is built for Kaggle with your local Qwen3 AWQ model:

`/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1`

Production design:

- **vLLM offline inference** for throughput
- Local **Qwen3-8B-AWQ** model
- Deterministic JSON generation
- Optional vLLM structured outputs
- JSON parse + schema validation
- Retry only invalid outputs
- Resume-safe checkpointing
- Kaggle 2×T4 fix: `disable_custom_all_reduce=True`

Important: on Kaggle 2×T4, vLLM tensor parallel startup may crash in custom all-reduce. This notebook disables custom all-reduce by default and can fall back to single-GPU vLLM if TP=2 still fails.


In [ ]:

# Cell 1 — Install runtime
# Run once. If Kaggle asks to restart after install, restart and continue from Cell 2.
# vLLM is used for high-throughput offline inference.
!pip -q install -U --no-cache-dir vllm transformers accelerate safetensors tqdm


In [ ]:

# Cell 2 — Imports and environment

from __future__ import annotations

import gc
import json
import os
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable, Iterator

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7")

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer

print("Python OK")
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} VRAM={p.total_memory/1024**3:.1f} GB")


In [ ]:

# Cell 3 — Configuration

IS_KAGGLE = Path("/kaggle/working").exists()

DATASET_SLUG = "swiss-court-authority-cards-rag-targets"
INPUT_FILENAME = "court_authority_cards_v4_target_cards.jsonl"

KAGGLE_MODEL_PATH = Path("/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1")
HF_FALLBACK_MODEL_ID = "Qwen/Qwen3-8B-AWQ"


def first_existing_input_file() -> Path:
    candidates = [
        Path(f"/kaggle/input/{DATASET_SLUG}/{INPUT_FILENAME}"),
        Path(f"/kaggle/input/datasets/samiulislam180041221/{DATASET_SLUG}/{INPUT_FILENAME}"),
        Path("/kaggle/input/datasets") / INPUT_FILENAME,
        Path("/kaggle/input") / INPUT_FILENAME,
        Path("./") / INPUT_FILENAME,
    ]
    for p in candidates:
        if p.exists():
            return p

    if Path("/kaggle/input").exists():
        found = sorted(Path("/kaggle/input").glob(f"**/{INPUT_FILENAME}"))
        if found:
            return found[0]

    return candidates[0]


def first_existing_model_source() -> str:
    if KAGGLE_MODEL_PATH.exists():
        return str(KAGGLE_MODEL_PATH)
    return HF_FALLBACK_MODEL_ID


@dataclass
class Config:
    # Files
    input_file: Path = first_existing_input_file()
    output_file: Path = Path("/kaggle/working/court_authority_cards_rag_targets_qwen3_8b_vllm.jsonl")
    failed_file: Path = Path("/kaggle/working/court_authority_cards_rag_targets_qwen3_8b_vllm_failed.jsonl")
    checkpoint_file: Path = Path("/kaggle/working/rag_targets_qwen3_8b_vllm_checkpoint.txt")

    # Model
    model_id: str = first_existing_model_source()
    quantization: str = "awq"
    dtype: str = "float16"
    max_model_len: int = 4096
    gpu_memory_utilization: float = 0.90

    # Tensor parallelism
    # Start with 2 on Kaggle 2xT4. If startup still fails, the loader can fall back to TP=1.
    tensor_parallel_size: int = 2

    # Kaggle 2xT4 reliability fix.
    # Your failure was in custom_all_reduce.cuh, so keep this True unless you are on a different GPU setup.
    disable_custom_all_reduce: bool = True
    auto_fallback_to_single_gpu: bool = True

    # Throughput controls
    # vLLM schedules internally; batch_size is how many prompts are submitted per notebook chunk.
    batch_size: int = 128
    max_num_seqs: int = 128

    # Prompt/output controls
    text_chars: int = 1600
    max_new_tokens: int = 224
    retry_max_new_tokens: int = 384

    # Accuracy/reliability controls
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.03
    main_enable_thinking: bool = False
    retry_enable_thinking: bool = True
    retry_invalid: bool = True
    use_structured_outputs: bool = True

    # Run controls
    # limit=50/500 for tests, limit=0 for full run.
    limit: int = 500
    start: int | None = None
    reset_output: bool = False
    flush_every_batches: int = 1

CONFIG = Config()

print("Input     :", CONFIG.input_file)
print("Output    :", CONFIG.output_file)
print("Failed    :", CONFIG.failed_file)
print("Checkpoint:", CONFIG.checkpoint_file)
print("Model     :", CONFIG.model_id)
print("TP size   :", CONFIG.tensor_parallel_size)
print("Disable custom all-reduce:", CONFIG.disable_custom_all_reduce)
print("Auto fallback TP=1:", CONFIG.auto_fallback_to_single_gpu)
print("Limit     :", CONFIG.limit)


In [ ]:

# Cell 4 — Smoke test

def count_lines(path: Path) -> int:
    n = 0
    with path.open("rb") as f:
        for _ in f:
            n += 1
    return n


print("=" * 72)
print("Smoke Test")
print("=" * 72)

assert CONFIG.input_file.exists(), f"Input file not found: {CONFIG.input_file}"
print("Input exists:", CONFIG.input_file)
print("Input size GB:", CONFIG.input_file.stat().st_size / 1024**3)
print("Input lines:", count_lines(CONFIG.input_file))

with CONFIG.input_file.open("r", encoding="utf-8") as f:
    first = json.loads(next(f))
print("First citation:", first.get("citation"))
print("Fields:", sorted(first.keys()))

if Path(CONFIG.model_id).exists():
    model_path = Path(CONFIG.model_id)
    print("Local model path exists:", model_path)
    for name in ["config.json", "tokenizer_config.json"]:
        print(name, "OK" if (model_path / name).exists() else "MISSING")
    shards = list(model_path.glob("*.safetensors"))
    print("Safetensors shards:", len(shards))
else:
    print("Model is HF ID:", CONFIG.model_id)

tok = AutoTokenizer.from_pretrained(CONFIG.model_id, trust_remote_code=True)
probe = tok("Swiss Federal Tribunal").input_ids
print("Tokenizer OK. Probe tokens:", len(probe))

msg = [
    {"role": "system", "content": "Return JSON only."},
    {"role": "user", "content": "Test."},
]
rendered = tok.apply_chat_template(
    msg,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("Chat template OK. Rendered chars:", len(rendered))

CONFIG.output_file.parent.mkdir(parents=True, exist_ok=True)
test_write = CONFIG.output_file.parent / "_write_test.tmp"
test_write.write_text("ok", encoding="utf-8")
test_write.unlink()
print("Working dir writable")

print("=" * 72)
print("Smoke test passed")
print("=" * 72)


In [ ]:

# Cell 5 — Schema, prompt, and validators

RAG_SCHEMA_HINT = {
    "english_summary": "string",
    "legal_topic": "string",
    "legal_question": "string",
    "legal_rule": "string",
    "court_holding": "string",
    "factual_context": "string",
    "english_legal_concepts": ["string"],
    "search_keywords": ["string"],
    "natural_language_queries": ["string"],
    "paragraph_role": "holding|reasoning|background|cost|procedural|disposition|standard_of_review|obiter",
    "outcome_signal": "granted|dismissed|inadmissible|remitted|partial|none",
}

REQUIRED_FIELDS = list(RAG_SCHEMA_HINT.keys())

ROLE_VALUES = {
    "holding",
    "reasoning",
    "background",
    "cost",
    "procedural",
    "disposition",
    "standard_of_review",
    "obiter",
}

OUTCOME_VALUES = {
    "granted",
    "dismissed",
    "inadmissible",
    "remitted",
    "partial",
    "none",
}

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "english_summary": {"type": "string"},
        "legal_topic": {"type": "string"},
        "legal_question": {"type": "string"},
        "legal_rule": {"type": "string"},
        "court_holding": {"type": "string"},
        "factual_context": {"type": "string"},
        "english_legal_concepts": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 8,
        },
        "search_keywords": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 12,
        },
        "natural_language_queries": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 5,
        },
        "paragraph_role": {"type": "string", "enum": sorted(ROLE_VALUES)},
        "outcome_signal": {"type": "string", "enum": sorted(OUTCOME_VALUES)},
    },
    "required": REQUIRED_FIELDS,
    "additionalProperties": False,
}

SYSTEM_PROMPT = (
    "You are a deterministic Swiss legal text-to-JSON extraction engine. "
    "The input is one paragraph from a Swiss court decision in German, French, or Italian, "
    "plus deterministic metadata. Return exactly one valid JSON object matching the requested schema. "
    "Use concise English legal terminology for semantic-search RAG. "
    "Use only statutes, articles, laws, and case citations present in the paragraph or metadata. "
    "Do not invent article numbers, laws, statutes, citations, facts, holdings, or outcomes. "
    "If a field is not supported, use an empty string, empty array, or 'none'. "
    "No markdown. No commentary. No explanations. No chain-of-thought."
)

REPAIR_GUARD = (
    "\n\nStrict repair instruction: return one valid compact JSON object only. "
    "No markdown fences, no prose, no explanation, no <think> block. "
    "All required keys must be present exactly once."
)


def safe_str(x: Any, max_chars: int = 1200) -> str:
    if x is None:
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:max_chars]


def safe_list(x: Any, max_items: int = 8, max_chars: int = 120) -> list[str]:
    if x is None:
        return []
    if not isinstance(x, list):
        x = [x]
    out = []
    seen = set()
    for item in x:
        s = safe_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def build_user_prompt(card: dict[str, Any], *, text_chars: int) -> str:
    text = safe_str(card.get("text_excerpt_original", ""), max_chars=text_chars)

    metadata = {
        "citation": card.get("citation") or "",
        "language": card.get("language") or "",
        "court_base": card.get("court_base") or "",
        "authority_role": card.get("authority_role") or "",
        "legal_area": card.get("legal_area") or "",
        "family": card.get("family") or "",
        "subfamily": card.get("subfamily") or "",
        "issue_labels_en": card.get("issue_labels_en") or [],
        "law_codes": card.get("law_codes") or [],
        "statutes_cited": card.get("statutes_cited") or [],
        "court_cases_cited": card.get("court_cases_cited") or [],
        "summary_en_proxy": card.get("summary_en_proxy") or "",
        "retrieval_text_en": card.get("retrieval_text_en") or "",
        "structural": card.get("structural") or {},
    }

    return (
        "Create RAG-target JSON for this court-authority paragraph.\n\n"
        "Required JSON schema keys:\n"
        f"{json.dumps(RAG_SCHEMA_HINT, ensure_ascii=False)}\n\n"
        "Rules:\n"
        "- Return JSON object only.\n"
        "- Keep strings concise and legally precise.\n"
        "- The natural_language_queries must be useful English search queries for retrieval.\n"
        "- Use paragraph_role and outcome_signal only from the allowed enum values.\n"
        "- Do not invent facts or legal authorities.\n\n"
        "Metadata:\n"
        f"{json.dumps(metadata, ensure_ascii=False, sort_keys=True)}\n\n"
        "Original paragraph:\n"
        f"{text}\n"
    )


def strip_think_blocks(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text.strip())
    return text.strip()


def extract_first_json_object(text: str) -> dict[str, Any]:
    text = strip_think_blocks(text)

    # Fast path.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Balanced-object extraction.
    start = text.find("{")
    if start < 0:
        raise ValueError("no JSON object start found")

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start : i + 1]
                    obj = json.loads(candidate)
                    if not isinstance(obj, dict):
                        raise ValueError("JSON root is not object")
                    return obj

    raise ValueError("no balanced JSON object found")


def normalize_and_validate(obj: dict[str, Any]) -> tuple[dict[str, Any], list[str]]:
    errors = []

    norm = {}
    for key in REQUIRED_FIELDS:
        if key not in obj:
            errors.append(f"missing:{key}")

    for key in [
        "english_summary",
        "legal_topic",
        "legal_question",
        "legal_rule",
        "court_holding",
        "factual_context",
    ]:
        norm[key] = safe_str(obj.get(key, ""), max_chars=1200)

    norm["english_legal_concepts"] = safe_list(obj.get("english_legal_concepts", []), max_items=8)
    norm["search_keywords"] = safe_list(obj.get("search_keywords", []), max_items=12)
    norm["natural_language_queries"] = safe_list(obj.get("natural_language_queries", []), max_items=5, max_chars=200)

    role = safe_str(obj.get("paragraph_role", "")).lower()
    if role not in ROLE_VALUES:
        errors.append(f"bad_paragraph_role:{role}")
        role = "reasoning"
    norm["paragraph_role"] = role

    outcome = safe_str(obj.get("outcome_signal", "")).lower()
    if outcome not in OUTCOME_VALUES:
        errors.append(f"bad_outcome_signal:{outcome}")
        outcome = "none"
    norm["outcome_signal"] = outcome

    # Minimum usefulness check.
    if not any(norm[k] for k in ["english_summary", "legal_question", "legal_rule", "court_holding"]):
        errors.append("empty_core_fields")

    if not norm["natural_language_queries"] and not norm["search_keywords"]:
        errors.append("empty_retrieval_terms")

    return norm, errors


def parse_validate_raw(raw: str) -> tuple[dict[str, Any] | None, list[str]]:
    try:
        obj = extract_first_json_object(raw)
    except Exception as exc:
        return None, [f"json_parse:{type(exc).__name__}:{str(exc)[:160]}"]

    norm, errors = normalize_and_validate(obj)
    if errors:
        return norm, errors
    return norm, []


In [ ]:

# Cell 6 — vLLM generator

from vllm import LLM, SamplingParams

try:
    from vllm.sampling_params import StructuredOutputsParams
except Exception:
    StructuredOutputsParams = None


class VLLMJsonGenerator:
    def __init__(self, config: Config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_id, trust_remote_code=True)

        if config.tensor_parallel_size > max(1, torch.cuda.device_count()):
            print(
                f"Requested TP={config.tensor_parallel_size}, but only {torch.cuda.device_count()} CUDA devices found. "
                "Using available device count."
            )
            config.tensor_parallel_size = max(1, torch.cuda.device_count())

        self.llm = self._load_llm_with_fallback(config)

        self.fast_params = self._make_params(
            max_tokens=config.max_new_tokens,
            structured=config.use_structured_outputs,
        )
        self.retry_params = self._make_params(
            max_tokens=config.retry_max_new_tokens,
            structured=config.use_structured_outputs,
        )

    def _llm_kwargs(self, config: Config) -> dict[str, Any]:
        return dict(
            model=config.model_id,
            quantization=config.quantization,
            dtype=config.dtype,
            trust_remote_code=True,
            tensor_parallel_size=config.tensor_parallel_size,
            gpu_memory_utilization=config.gpu_memory_utilization,
            max_model_len=config.max_model_len,
            max_num_seqs=config.max_num_seqs,
            enable_prefix_caching=True,
            disable_log_stats=True,

            # Critical Kaggle 2xT4 fix:
            # prevents vLLM custom_all_reduce.cuh invalid-argument startup failures.
            disable_custom_all_reduce=config.disable_custom_all_reduce,
        )

    def _load_llm_with_fallback(self, config: Config) -> LLM:
        print("[vLLM] loading model:", config.model_id)
        print("[vLLM] quantization:", config.quantization)
        print("[vLLM] tensor_parallel_size:", config.tensor_parallel_size)
        print("[vLLM] disable_custom_all_reduce:", config.disable_custom_all_reduce)

        try:
            return LLM(**self._llm_kwargs(config))
        except Exception as exc:
            print("[vLLM] initial load failed:", repr(exc))

            if not config.auto_fallback_to_single_gpu or config.tensor_parallel_size <= 1:
                raise

            print("[vLLM] falling back to tensor_parallel_size=1.")
            print("[vLLM] If this fallback also fails, restart the Kaggle session and run with TP=1 from the start.")

            # Best-effort cleanup before retrying.
            try:
                del self.llm
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            config.tensor_parallel_size = 1
            config.max_num_seqs = min(config.max_num_seqs, 96)
            config.batch_size = min(config.batch_size, 96)
            return LLM(**self._llm_kwargs(config))

    def _make_params(self, *, max_tokens: int, structured: bool) -> SamplingParams:
        kwargs = dict(
            temperature=self.config.temperature,
            top_p=self.config.top_p,
            max_tokens=max_tokens,
            repetition_penalty=self.config.repetition_penalty,
            stop=["<|im_end|>", "</s>"],
        )

        if structured and StructuredOutputsParams is not None:
            try:
                kwargs["structured_outputs"] = StructuredOutputsParams(json=JSON_SCHEMA)
                print("[vLLM] structured JSON outputs enabled")
            except Exception as exc:
                print("[vLLM] structured outputs unavailable, falling back to parser:", repr(exc))

        return SamplingParams(**kwargs)

    def render_prompt(self, card: dict[str, Any], *, enable_thinking: bool, extra_guard: str = "") -> str:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": build_user_prompt(card, text_chars=self.config.text_chars) + extra_guard,
            },
        ]

        try:
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        except TypeError:
            # Older tokenizer fallback.
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

    def generate_raw(
        self,
        cards: list[dict[str, Any]],
        *,
        retry: bool = False,
        extra_guard: str = "",
    ) -> list[str]:
        enable_thinking = self.config.retry_enable_thinking if retry else self.config.main_enable_thinking
        params = self.retry_params if retry else self.fast_params

        prompts = [
            self.render_prompt(card, enable_thinking=enable_thinking, extra_guard=extra_guard)
            for card in cards
        ]

        outputs = self.llm.generate(prompts, sampling_params=params, use_tqdm=False)
        return [out.outputs[0].text for out in outputs]


In [ ]:

# Cell 7 — IO, checkpointing, and main loop

def read_checkpoint(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        return int(path.read_text(encoding="utf-8").strip() or "0")
    except Exception:
        return 0


def write_checkpoint(path: Path, next_line_idx: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(str(next_line_idx), encoding="utf-8")


def iter_cards(path: Path, *, start: int, stop_before: int | None) -> Iterator[tuple[int, dict[str, Any]]]:
    with path.open("r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx < start:
                continue
            if stop_before is not None and idx >= stop_before:
                break
            line = line.strip()
            if not line:
                continue
            try:
                card = json.loads(line)
            except Exception as exc:
                yield idx, {"_read_error": repr(exc), "_raw_line": line[:1000]}
                continue
            yield idx, card


def make_output_record(
    *,
    line_idx: int,
    card: dict[str, Any],
    rag_json: dict[str, Any] | None,
    raw: str,
    errors: list[str],
    source: str,
    elapsed_s: float,
) -> dict[str, Any]:
    rec = dict(card)
    rec["_source_line_idx"] = line_idx
    rec["_rag_generation"] = {
        "model": CONFIG.model_id,
        "engine": "vllm",
        "source": source,
        "elapsed_s": round(elapsed_s, 4),
        "errors": errors,
    }
    rec["rag_targets_qwen3_8b_awq"] = rag_json
    if errors:
        rec["_raw_model_output"] = raw[:4000]
    return rec


def run_one_batch(
    generator: VLLMJsonGenerator,
    batch_pairs: list[tuple[int, dict[str, Any]]],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, int | float]]:
    batch_cards = [card for _, card in batch_pairs]
    t0 = time.time()

    raws = generator.generate_raw(batch_cards, retry=False)
    first_elapsed = time.time() - t0

    parsed = []
    retry_indices = []

    for j, raw in enumerate(raws):
        norm, errors = parse_validate_raw(raw)
        parsed.append((norm, errors, raw, "main"))
        if errors and CONFIG.retry_invalid:
            retry_indices.append(j)

    retry_elapsed = 0.0
    if retry_indices:
        retry_cards = [batch_cards[j] for j in retry_indices]
        t1 = time.time()
        retry_raws = generator.generate_raw(retry_cards, retry=True, extra_guard=REPAIR_GUARD)
        retry_elapsed = time.time() - t1

        for local_k, j in enumerate(retry_indices):
            raw2 = retry_raws[local_k]
            norm2, errors2 = parse_validate_raw(raw2)
            # Keep retry if it fully validates or improves parseability.
            old_norm, old_errors, old_raw, old_source = parsed[j]
            if not errors2 or (old_norm is None and norm2 is not None):
                parsed[j] = (norm2, errors2, raw2, "retry")
            else:
                parsed[j] = (old_norm, old_errors, old_raw, old_source)

    ok_records = []
    failed_records = []

    total_elapsed = first_elapsed + retry_elapsed
    per_item_elapsed = total_elapsed / max(len(batch_pairs), 1)

    for (line_idx, card), (norm, errors, raw, source) in zip(batch_pairs, parsed):
        rec = make_output_record(
            line_idx=line_idx,
            card=card,
            rag_json=norm,
            raw=raw,
            errors=errors,
            source=source,
            elapsed_s=per_item_elapsed,
        )
        if errors:
            failed_records.append(rec)
        else:
            ok_records.append(rec)

    stats = {
        "batch_size": len(batch_pairs),
        "ok": len(ok_records),
        "failed": len(failed_records),
        "retried": len(retry_indices),
        "elapsed_s": total_elapsed,
        "cards_s": len(batch_pairs) / max(total_elapsed, 1e-9),
    }
    return ok_records, failed_records, stats


def append_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    if not records:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False, separators=(",", ":")) + "\n")
        f.flush()
        os.fsync(f.fileno())


def main(config: Config = CONFIG) -> None:
    total = count_lines(config.input_file)

    if config.reset_output:
        for p in [config.output_file, config.failed_file, config.checkpoint_file]:
            if p.exists():
                p.unlink()

    start = config.start if config.start is not None else read_checkpoint(config.checkpoint_file)
    if start < 0:
        start = 0

    stop_before = None if config.limit == 0 else min(total, start + config.limit)

    print("Input      :", config.input_file)
    print("Output     :", config.output_file)
    print("Failures   :", config.failed_file)
    print("Checkpoint :", config.checkpoint_file)
    print("Model      :", config.model_id)
    print("Start      :", start)
    print("Total      :", total)
    print("To process :", "all remaining" if stop_before is None else max(stop_before - start, 0))
    print("Batch size :", config.batch_size)
    print("Structured :", config.use_structured_outputs)
    print("Thinking   :", {"main": config.main_enable_thinking, "retry": config.retry_enable_thinking})

    generator = VLLMJsonGenerator(config)

    pending: list[tuple[int, dict[str, Any]]] = []
    processed = 0
    ok_total = 0
    fail_total = 0
    retry_total = 0
    t_run = time.time()
    last_line_idx = start

    total_for_bar = None if stop_before is None else max(stop_before - start, 0)
    pbar = tqdm(total=total_for_bar, desc="vllm-json", unit="card")

    def flush_batch():
        nonlocal pending, processed, ok_total, fail_total, retry_total, last_line_idx
        if not pending:
            return

        batch_pairs = pending
        pending = []

        ok_records, failed_records, stats = run_one_batch(generator, batch_pairs)
        append_jsonl(config.output_file, ok_records)
        append_jsonl(config.failed_file, failed_records)

        last_line_idx = batch_pairs[-1][0] + 1
        write_checkpoint(config.checkpoint_file, last_line_idx)

        processed += len(batch_pairs)
        ok_total += len(ok_records)
        fail_total += len(failed_records)
        retry_total += int(stats["retried"])

        if pbar:
            pbar.update(len(batch_pairs))

        elapsed = time.time() - t_run
        avg = processed / max(elapsed, 1e-9)
        remaining = None
        eta_h = None
        if stop_before is not None:
            remaining = max(stop_before - last_line_idx, 0)
            eta_h = remaining / max(avg, 1e-9) / 3600

        print(
            f"[batch] size={stats['batch_size']} ok={stats['ok']} failed={stats['failed']} "
            f"retried={stats['retried']} batch_rate={stats['cards_s']:.2f} cards/s "
            f"avg={avg:.2f} cards/s eta_h={eta_h if eta_h is not None else 'full'} "
            f"checkpoint={last_line_idx}",
            flush=True,
        )

    try:
        for line_idx, card in iter_cards(config.input_file, start=start, stop_before=stop_before):
            pending.append((line_idx, card))
            if len(pending) >= config.batch_size:
                flush_batch()
        flush_batch()
    finally:
        if pbar:
            pbar.close()

    elapsed = time.time() - t_run
    avg = processed / max(elapsed, 1e-9)
    full_remaining = max(total - last_line_idx, 0)
    full_eta_h = full_remaining / max(avg, 1e-9) / 3600

    print("=" * 72)
    print("Done")
    print("Processed       :", processed)
    print("OK              :", ok_total)
    print("Failed          :", fail_total)
    print("Retried         :", retry_total)
    print("Elapsed min     :", elapsed / 60)
    print("Average cards/s :", avg)
    print("Checkpoint      :", read_checkpoint(config.checkpoint_file))
    print("Full-run ETA h  :", full_eta_h)
    print("=" * 72)


In [ ]:

# Cell 8 — Run a quality/throughput test
# Recommended first run:
#   CONFIG.limit = 500
#   CONFIG.batch_size = 128
#   CONFIG.max_num_seqs = 128
# After quality checks:
#   CONFIG.limit = 0
#   CONFIG.reset_output = False

CONFIG.limit = 500
CONFIG.batch_size = 128
CONFIG.max_num_seqs = 128
CONFIG.tensor_parallel_size = 2
CONFIG.disable_custom_all_reduce = True
CONFIG.auto_fallback_to_single_gpu = True
CONFIG.reset_output = False

main(CONFIG)


In [ ]:

# Cell 9 — Inspect output quality

def read_jsonl_tail(path: Path, n: int = 3) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8").splitlines()
    out = []
    for line in lines[-n:]:
        if line.strip():
            out.append(json.loads(line))
    return out


samples = read_jsonl_tail(CONFIG.output_file, 3)
for i, rec in enumerate(samples, 1):
    print("=" * 72)
    print("Sample", i)
    print("citation:", rec.get("citation"))
    print("generation:", rec.get("_rag_generation"))
    print(json.dumps(rec.get("rag_targets_qwen3_8b_awq"), ensure_ascii=False, indent=2))


In [ ]:

# Cell 10 — Full production run
# Run this only after inspecting quality from Cell 9.

# CONFIG.limit = 0
# CONFIG.reset_output = False
# main(CONFIG)


In [ ]:

# Cell 11 — Optional tuning presets

# Higher accuracy, slower:
# CONFIG.text_chars = 2200
# CONFIG.max_new_tokens = 320
# CONFIG.retry_max_new_tokens = 512
# CONFIG.retry_enable_thinking = True
# CONFIG.batch_size = 96
# CONFIG.max_num_seqs = 96

# Faster, still reliable if output quality passes:
# CONFIG.text_chars = 1200
# CONFIG.max_new_tokens = 192
# CONFIG.retry_max_new_tokens = 320
# CONFIG.retry_enable_thinking = False
# CONFIG.batch_size = 192
# CONFIG.max_num_seqs = 192

# If Kaggle 2xT4 has vLLM tensor-parallel issues:
# CONFIG.tensor_parallel_size = 1
# CONFIG.batch_size = 96
# CONFIG.max_num_seqs = 96

# Kaggle 2xT4 stable vLLM startup preset:
# CONFIG.quantization = "awq"
# CONFIG.tensor_parallel_size = 2
# CONFIG.disable_custom_all_reduce = True
# CONFIG.auto_fallback_to_single_gpu = True
# CONFIG.batch_size = 128
# CONFIG.max_num_seqs = 128

# If TP=2 still fails after disabling custom all-reduce, restart the Kaggle session and use:
# CONFIG.tensor_parallel_size = 1
# CONFIG.batch_size = 96
# CONFIG.max_num_seqs = 96
# main(CONFIG)

# Optional speed experiment only after a successful 500-card test:
# CONFIG.quantization = "awq_marlin"
# CONFIG.tensor_parallel_size = 1
# CONFIG.batch_size = 128
# CONFIG.max_num_seqs = 128
# main(CONFIG)
